In [1]:
# ============================================================
# Notebook    : 02_sequence_builder.ipynb
# Description : Build sequences and establish the IDpol-level
#               train/val/test split used by ALL downstream
#               notebooks (03a/03b LightGBM, 04a/04b Transformer).
#               - Integer-index categorical encoding (PAD=0)
#               - Feature scope: original 5 variables only
#                 (Expo, YearGap, Usage, VehType, VehPower)
#                 Aggregate/history features deferred to a later
#                 notebook (not this one, per design decision)
#               - Split ratio 70/15/15, seed=42 (unchanged from
#                 prior design)
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install torch tqdm pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import pandas as pd
import numpy as np
import torch
import json
import os
import pickle
from tqdm import tqdm

os.makedirs("data/sequences", exist_ok=True)

SEED = 42
np.random.seed(SEED)


# ============================================================
# 2. Load preprocessed multi-history data (from notebook 01)
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_features.csv")

print(f"[CHECK 1] df shape: {df.shape}")
print(f"[CHECK 1] Unique IDpol: {df['IDpol'].nunique():,}")
print(df.head())


# ============================================================
# 3. Feature scope — ORIGINAL 5 VARIABLES ONLY
#    - Numeric   : Expo, YearGap
#    - Categorical: Usage, VehType, VehPower
#    - PrevLabel / LabelChanged / NbClaim are NOT feature inputs
#      (label / EDA-only columns, same as prior design)
# ============================================================
NUMERIC_COLS = ["Expo", "YearGap"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
LABEL_COL    = "Label"

print(f"\n[CHECK 2] Feature scope confirmed:")
print(f"  Numeric     : {NUMERIC_COLS}")
print(f"  Categorical : {CAT_COLS}")
print(f"  Label       : {LABEL_COL}")
print(f"  (Aggregate/history features deferred to a later notebook)")


# ============================================================
# 4. Build categorical vocabularies
#    - Index 0 reserved for padding token (PAD)
#    - Saved to json for reproducibility, and for 03a/03b to
#      build a matching one-hot / native-categorical encoding
# ============================================================
vocabs = {}

for col in CAT_COLS:
    categories = sorted(df[col].dropna().unique().tolist())
    vocab = {"<PAD>": 0}
    vocab.update({cat: i + 1 for i, cat in enumerate(categories)})
    vocabs[col] = vocab
    df[col + "_idx"] = df[col].map(vocab)

print("\n[CHECK 3] Vocabulary sizes (incl. PAD):")
for col, vocab in vocabs.items():
    print(f"  {col:12s} | {len(vocab)}")

with open("data/sequences/vocabs.json", "w", encoding="utf-8") as f:
    json.dump(vocabs, f, ensure_ascii=False, indent=2)
print("Saved: data/sequences/vocabs.json")


# ============================================================
# 5. IDpol-level train / val / test split
#    - 70 / 15 / 15, seed=42 — UNCHANGED from prior design
#    - This split is the single source of truth for every
#      downstream notebook (03a, 03b, 04a, 04b). All four
#      models MUST evaluate on the same test IDpol population.
# ============================================================
unique_ids = df["IDpol"].unique()
rng = np.random.RandomState(SEED)
rng.shuffle(unique_ids)

n = len(unique_ids)
train_ids = unique_ids[: int(n * 0.7)]
val_ids   = unique_ids[int(n * 0.7): int(n * 0.85)]
test_ids  = unique_ids[int(n * 0.85):]

print(f"\n[CHECK 4] Split sizes (policyholders):")
print(f"  Train: {len(train_ids):>7,}")
print(f"  Val  : {len(val_ids):>7,}")
print(f"  Test : {len(test_ids):>7,}")

# sanity: no overlap
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
print(f"[CHECK 4] Overlap check passed (0/0/0)")

# Save IDpol split itself — 03a/03b (LightGBM) read this directly
# without needing the pickled sequences
split_ids = {
    "train": train_ids.tolist(),
    "val":   val_ids.tolist(),
    "test":  test_ids.tolist(),
}
with open("data/sequences/split_idpols.json", "w", encoding="utf-8") as f:
    json.dump(split_ids, f)
print("Saved: data/sequences/split_idpols.json")


# ============================================================
# 6. Build variable-length sequence tensors per IDpol
#    - Used by 04a/04b (Transformer). NOT used by 03a/03b.
# ============================================================
CAT_IDX_COLS = [c + "_idx" for c in CAT_COLS]

df_sorted = df.sort_values(["IDpol", "Year"])
grouped_data = {k: v for k, v in df_sorted.groupby("IDpol")}

def build_sequences(ids, grouped_dict, desc="building"):
    sequences = []
    for idpol in tqdm(ids, desc=desc):
        sub = grouped_dict[idpol]
        seq = {
            "IDpol"    : idpol,
            "length"   : len(sub),
            "numeric"  : sub[NUMERIC_COLS].values.astype(np.float32),
            "cat_idx"  : sub[CAT_IDX_COLS].values.astype(np.int64),
            "label"    : sub[LABEL_COL].values.astype(np.int64),
            "year"     : sub["Year"].values.astype(np.int64),
        }
        sequences.append(seq)
    return sequences

train_seqs = build_sequences(train_ids, grouped_data, desc="train")
val_seqs   = build_sequences(val_ids,   grouped_data, desc="val")
test_seqs  = build_sequences(test_ids,  grouped_data, desc="test")

print(f"\n[CHECK 5] Sequences built:")
print(f"  Train: {len(train_seqs):>7,}")
print(f"  Val  : {len(val_seqs):>7,}")
print(f"  Test : {len(test_seqs):>7,}")


# ============================================================
# 7. Save sequence objects to disk (pickle)
# ============================================================
for name, seqs in [("train", train_seqs), ("val", val_seqs), ("test", test_seqs)]:
    with open(f"data/sequences/{name}_sequences.pkl", "wb") as f:
        pickle.dump(seqs, f)
    print(f"Saved: data/sequences/{name}_sequences.pkl")


# ============================================================
# 8. Summary check
# ============================================================
vocab_sizes = {k: len(v) for k, v in vocabs.items()}

print("\n===== Sequence Builder Summary =====")
print(f"Feature scope        : {NUMERIC_COLS + CAT_COLS} (original 5 vars only)")
print(f"Vocab sizes          : {vocab_sizes}")
print(f"Split (train/val/test): {len(train_ids):,} / {len(val_ids):,} / {len(test_ids):,} (70/15/15, seed={SEED})")
print(f"IDpol overlap         : 0 / 0 / 0 (verified)")
print(f"Outputs               : split_idpols.json, vocabs.json, {{train,val,test}}_sequences.pkl")
print("=====================================")

[CHECK 1] df shape: (364997, 11)
[CHECK 1] Unique IDpol: 71,392
      IDpol  Year  NbClaim      Expo    Usage   VehType   VehPower  Label  \
0  PN100006  2003        0  0.669399  Usage15  VehType8  VehPower1      0   
1  PN100006  2004        0  0.997268  Usage15  VehType8  VehPower1      0   
2  PN100006  2005        0  0.997268  Usage15  VehType8  VehPower1      0   
3  PN100006  2006        0  0.997268  Usage15  VehType8  VehPower1      0   
4  PN100006  2007        0  1.000000  Usage15  VehType8  VehPower1      0   

   YearGap  PrevLabel  LabelChanged  
0        0        NaN           NaN  
1        1        0.0           0.0  
2        2        0.0           0.0  
3        3        0.0           0.0  
4        4        0.0           0.0  

[CHECK 2] Feature scope confirmed:
  Numeric     : ['Expo', 'YearGap']
  Categorical : ['Usage', 'VehType', 'VehPower']
  Label       : Label
  (Aggregate/history features deferred to a later notebook)

[CHECK 3] Vocabulary sizes (incl. PAD):
 

test: 100%|█████████████████████████████████████████████████████████████████████| 10709/10709 [00:11<00:00, 927.56it/s]



[CHECK 5] Sequences built:
  Train:  49,974
  Val  :  10,709
  Test :  10,709
Saved: data/sequences/train_sequences.pkl
Saved: data/sequences/val_sequences.pkl
Saved: data/sequences/test_sequences.pkl

===== Sequence Builder Summary =====
Feature scope        : ['Expo', 'YearGap', 'Usage', 'VehType', 'VehPower'] (original 5 vars only)
Vocab sizes          : {'Usage': 19, 'VehType': 16, 'VehPower': 9}
Split (train/val/test): 49,974 / 10,709 / 10,709 (70/15/15, seed=42)
IDpol overlap         : 0 / 0 / 0 (verified)
Outputs               : split_idpols.json, vocabs.json, {train,val,test}_sequences.pkl
